# Random Forest Baseline for Shape Classification

This notebook loads the master trial dataset, builds the raw feature matrix, applies the train/validation/test split, and evaluates the initial Random Forest model.

## 1. Import the libraries needed for data loading, splitting, and evaluation.
This cell imports pandas, the Random Forest model, and the metrics used to assess validation performance.

In [12]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## 2. Load the master trial dataset and define the sensor feature columns.
This cell reads the full labeled dataset from the raw CSV and identifies the 12 resistance features we will use as inputs.

In [13]:
df = pd.read_csv('../../data/trial_data.csv')
feature_cols = [f'r{i}' for i in range(1, 13)]
df.head()

,trial_ID,r1,r2,r3,r4,r5,r6,r7,r8,r9,r10,r11,r12,shape
0,1,1.001244,1.002524,0.995667,0.968140,1.000352,1.003129,0.999661,0.977056,1.000150,1.000891,0.998214,0.992309,round_right
1,2,1.001555,1.002524,0.995999,0.966800,1.000175,1.002782,1.000678,0.977533,1.000299,1.000742,0.998362,0.994669,round_right
2,3,1.001711,1.002694,0.996162,0.967088,1.000000,1.002956,1.000846,0.976831,1.000000,1.000595,0.998511,0.994670,round_right
3,4,1.001557,1.002524,0.996161,0.968531,1.000175,1.002608,1.000508,0.979040,1.000150,1.000595,0.999404,0.995704,round_right
4,5,1.001869,1.002863,0.995826,0.966419,1.000353,1.002959,1.000848,0.976813,1.000149,1.000593,0.998510,0.995113,round_right


## 3. Create the train, validation, and test splits.
This cell splits the full dataset proportionally into training, validation, and test samples while preserving class balance with stratification. The exact counts adjust automatically when rows are added.

In [14]:
X = df[feature_cols]
y = df['shape']
class_labels = sorted(y.dropna().unique())

train_idx, temp_idx = train_test_split(
    df.index.to_numpy(),
    test_size=0.32,
    stratify=y,
    random_state=RANDOM_STATE,
)

temp_df = df.loc[temp_idx].copy()
temp_y = temp_df['shape']
val_idx, test_idx = train_test_split(
    temp_df.index.to_numpy(),
    test_size=0.5,
    stratify=temp_y,
    random_state=RANDOM_STATE,
)

train_df = df.loc[train_idx].copy()
val_df = df.loc[val_idx].copy()
test_df = df.loc[test_idx].copy()

print(f'Total: {len(df)}')
print(f'Train: {len(train_df)} ({len(train_df) / len(df):.1%})')
print(f'Validation: {len(val_df)} ({len(val_df) / len(df):.1%})')
print(f'Test: {len(test_df)} ({len(test_df) / len(df):.1%})')
print(f'Split total: {len(train_df) + len(val_df) + len(test_df)}')

Total: 125
Train: 85 (68.0%)
Validation: 20 (16.0%)
Test: 20 (16.0%)
Split total: 125


## 4. Train the baseline Random Forest on the training set.
This cell creates the baseline model and fits it using only the train split; the validation data is kept separate for model selection.

In [15]:
model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
X_train = train_df[feature_cols]
y_train = train_df['shape']
X_val = val_df[feature_cols]
y_val = val_df['shape']
model.fit(X_train, y_train)
pred = model.predict(X_val)

## 5. Measure validation performance.
This cell prints the validation accuracy, classification report, and confusion matrix so we can judge how well the baseline model generalizes.

In [16]:
print('Validation accuracy:', (pred == y_val).mean())
print(classification_report(y_val, pred, labels=class_labels, digits=4))
print(confusion_matrix(y_val, pred, labels=class_labels))

Validation accuracy: 1.0
              precision    recall  f1-score   support

round_bottom     1.0000    1.0000    1.0000         4
  round_left     1.0000    1.0000    1.0000         4
 round_right     1.0000    1.0000    1.0000         4
   round_top     1.0000    1.0000    1.0000         4
  shape_none     1.0000    1.0000    1.0000         4

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.0000        20
weighted avg     1.0000    1.0000    1.0000        20

[[4 0 0 0 0]
 [0 4 0 0 0]
 [0 0 4 0 0]
 [0 0 0 4 0]
 [0 0 0 0 4]]


## 6. Hyperparameter tuning on the validation set
This cell runs a validation-based grid search for Random Forest hyperparameters. It creates a dictionary of candidate values for `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, and `class_weight`, then uses `itertools.product` to test every combination. For each combination, it trains a Random Forest on the training set, predicts the validation set, and computes validation accuracy and macro F1. The results are stored in `results`, sorted by validation macro F1, and the top 10 models are printed. This is the model-selection step: we are not using the test set yet. The first row printed is the best validation model, and that model is later used for the final hold-out test evaluation.

In [ ]:
from itertools import product

param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 4, 6, 8],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': [None, 'balanced']
}
results = []
keys = list(param_grid.keys())
values = list(param_grid.values())
for combo in product(*values):
    params = dict(zip(keys, combo))
    model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params)
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    val_accuracy = (pred_val == y_val).mean()
    val_macro_f1 = classification_report(y_val, pred_val, labels=class_labels, output_dict=True, zero_division=0)['macro avg']['f1-score']
    results.append({**params, 'validation_accuracy': val_accuracy, 'validation_macro_f1': val_macro_f1})
results_sorted = sorted(results, key=lambda x: x['validation_macro_f1'], reverse=True)
print(f'Total evaluated models: {len(results_sorted)}')
for rank, row in enumerate(results_sorted, start=1):
    print(f'Rank {rank}: {row}')

## 7. Select the final model parameters
Enter the hyperparameter values you want to use in the code cell below. These values are used to build and evaluate the final Random Forest model. Choose them based on the validation results above, before evaluating on the untouched test set.

In [ ]:
final_model_parameters = {
    'n_estimators': 200,
    'max_depth': None,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

## 8. Retrain the final model on the training and validation sets
After selecting the final hyperparameters, combine the training and validation data and fit a fresh model. The test set remains untouched until the next section.

In [ ]:
X_train_validation = pd.concat([train_df[feature_cols], val_df[feature_cols]], ignore_index=True)
y_train_validation = pd.concat([train_df['shape'], val_df['shape']], ignore_index=True)
final_model = RandomForestClassifier(**final_model_parameters)
final_model.fit(X_train_validation, y_train_validation)
print(f'Retrained final model on {len(X_train_validation)} training and validation samples.')

## 9. Final hold-out test evaluation (run once at the very end)
This is the final unbiased check. Only run this cell after the validation tuning loop and final retraining are complete. The test set has not been used to fit the model.

In [ ]:
X_test = test_df[feature_cols]
y_test = test_df['shape']
pred_test = final_model.predict(X_test)
print('Selected final model configuration:')
print(final_model_parameters)
print('Test accuracy:', (pred_test == y_test).mean())
print(classification_report(y_test, pred_test, labels=class_labels, digits=4, zero_division=0))
print(confusion_matrix(y_test, pred_test, labels=class_labels))

## 10. Save the final model
Run this cell after the final test evaluation to save the selected model for use by the GUI.

In [ ]:
from pathlib import Path
import joblib

saved_model_name = 'final_shape_classifier.joblib'  # change this name depending on the model
model_path = Path('../models') / saved_model_name
if model_path.exists():
    raise FileExistsError(f'Model file already exists: {model_path}. Choose a different saved_model_name.')
joblib.dump(final_model, model_path)
print(f'Model saved to: {model_path.resolve()}')